# 11. Baseline Models: Logistic Regression and Random Forest

This notebook trains logistic regression and random forest baselines across all four feature tiers, using the same train/test split as the XGBoost models. One-hot encoding is applied globally to all integer columns with more than 2 unique values, computed on training data only to avoid look-ahead bias. Float columns are left as-is (treated as continuous). Binary integer columns (exactly 2 unique values) are left as-is.

In [2]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv('hsls_clean.csv')

with open('train_test_split.json', 'r') as f:
    split = json.load(f)

with open('feature_tiers.json', 'r') as f:
    feature_tiers = json.load(f)

train_idx = split['train']
test_idx = split['test']

In [6]:
tier4_cols = feature_tiers['tier4']

# Floats that contain only whole numbers (NaN-coerced integers from cleaning)
whole_number_floats = [col for col in tier4_cols 
                       if col in df.columns 
                       and df[col].dtype == 'float64'
                       and df[col].dropna().apply(lambda x: x == int(x)).all()]

# True integers still in int dtype
true_integers = [col for col in tier4_cols 
                 if col in df.columns 
                 and df[col].dtype in ['int64', 'int32']]

# Combine and identify nominals: more than 2 unique non-NaN values
candidate_cols = whole_number_floats + true_integers

nominal_cols = [col for col in candidate_cols 
                if df[col].nunique() > 2]

binary_cols = [col for col in candidate_cols 
               if df[col].nunique() == 2]

# Remaining floats are genuinely continuous
continuous_float_cols = [col for col in tier4_cols 
                         if col in df.columns 
                         and col not in candidate_cols]

print(f"Nominal (to encode):       {len(nominal_cols)}")
print(f"Binary (leave as-is):      {len(binary_cols)}")
print(f"Continuous float (as-is):  {len(continuous_float_cols)}")
print(f"Total tier 4 features:     {len(tier4_cols)}")

Nominal (to encode):       418
Binary (leave as-is):      380
Continuous float (as-is):  35
Total tier 4 features:     833


In [7]:
print(continuous_float_cols)

['X1TXMTH', 'X1TXMSEM', 'X1TXMSCR', 'X1TXMTSCOR', 'X1TXMPROF1', 'X1TXMPROF2', 'X1TXMPROF3', 'X1TXMPROF4', 'X1TXMPROF5', 'X1SES', 'X1SES_U', 'X1MTHID', 'X1MTHUTI', 'X1MTHEFF', 'X1MTHINT', 'X1SCIID', 'X1SCIUTI', 'X1SCIEFF', 'X1SCIINT', 'X1SCHOOLBEL', 'X1SCHOOLENG', 'X1TMCOMM', 'X1TMEFF', 'X1TMEXP', 'X1TMPRINC', 'X1TMRESP', 'X1TSCOMM', 'X1TSEFF', 'X1TSEXP', 'X1TSPRINC', 'X1TSRESP', 'X1SCHOOLCLI', 'X1COUPERTEA', 'X1COUPERCOU', 'X1COUPERPRI']


In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols)
    ],
    remainder='passthrough'
)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

results = {}

for tier_name, tier_cols in feature_tiers.items():
    print(f"\n{'='*60}")
    print(f"TIER: {tier_name.upper()} — {len(tier_cols)} features")
    print(f"{'='*60}")

    tier_nominal    = [col for col in nominal_cols if col in tier_cols]
    tier_binary     = [col for col in binary_cols if col in tier_cols]
    tier_continuous = [col for col in continuous_float_cols if col in tier_cols]

    X_train_raw = df[tier_cols].iloc[train_idx]
    X_test_raw  = df[tier_cols].iloc[test_idx]
    y_train     = df['X4EVERDROP'].iloc[train_idx]
    y_test      = df['X4EVERDROP'].iloc[test_idx]

    # Shared preprocessor for both models
    preprocessor = ColumnTransformer(transformers=[
        ('nom', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), tier_nominal),
        ('bin', SimpleImputer(strategy='most_frequent'), tier_binary),
        ('con', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler())
        ]), tier_continuous),
    ])

    X_train = preprocessor.fit_transform(X_train_raw)
    X_test  = preprocessor.transform(X_test_raw)

    print(f"Encoded dimensions — train: {X_train.shape}, test: {X_test.shape}")

    tier_results = {}

    for model_name, model in [
        ('Logistic Regression', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
        ('Random Forest',       RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample',
                                                       max_depth=10, random_state=42))
    ]:
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        acc       = accuracy_score(y_test, y_pred)
        auc       = roc_auc_score(y_test, y_proba)
        report    = classification_report(y_test, y_pred, output_dict=True)
        f1        = report['1']['f1-score']
        precision = report['1']['precision']
        recall    = report['1']['recall']

        print(f"\n  {model_name}")
        print(f"  Accuracy: {acc:.4f} | AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

        tier_results[model_name] = {
            'accuracy': acc, 'auc': auc, 'f1': f1,
            'precision': precision, 'recall': recall
        }

    results[tier_name] = tier_results


TIER: TIER1 — 130 features
Encoded dimensions — train: (12654, 575), test: (3164, 575)

  Logistic Regression
  Accuracy: 0.7892 | AUC: 0.8297 | F1: 0.5004 | Precision: 0.3870 | Recall: 0.7076

  Random Forest
  Accuracy: 0.7895 | AUC: 0.8249 | F1: 0.4947 | Precision: 0.3853 | Recall: 0.6907

TIER: TIER2 — 192 features
Encoded dimensions — train: (12654, 1191), test: (3164, 1191)

  Logistic Regression
  Accuracy: 0.7797 | AUC: 0.8250 | F1: 0.4886 | Precision: 0.3737 | Recall: 0.7055

  Random Forest
  Accuracy: 0.7506 | AUC: 0.7986 | F1: 0.4555 | Precision: 0.3378 | Recall: 0.6992

TIER: TIER3 — 436 features
Encoded dimensions — train: (12654, 1842), test: (3164, 1842)

  Logistic Regression
  Accuracy: 0.7857 | AUC: 0.8187 | F1: 0.4752 | Precision: 0.3744 | Recall: 0.6504

  Random Forest
  Accuracy: 0.8040 | AUC: 0.8154 | F1: 0.4833 | Precision: 0.3984 | Recall: 0.6144

TIER: TIER4 — 833 features
Encoded dimensions — train: (12654, 3230), test: (3164, 3230)

  Logistic Regression
 

In [13]:
import pandas as pd

xgb_results = {
    'tier1': {'f1': 0.5371, 'auc': 0.8494, 'precision': 0.4606, 'recall': 0.6441},
    'tier2': {'f1': 0.5400, 'auc': 0.8534, 'precision': 0.4817, 'recall': 0.6144},
    'tier3': {'f1': 0.5546, 'auc': 0.8632, 'precision': 0.4799, 'recall': 0.6568},
    'tier4': {'f1': 0.5582, 'auc': 0.8658, 'precision': 0.4754, 'recall': 0.6758},
}

rows = []
for tier_name in feature_tiers.keys():
    tier_label = tier_name.capitalize()
    n_features = len(feature_tiers[tier_name])
    for model_name in ['Logistic Regression', 'Random Forest']:
        r = results[tier_name][model_name]
        rows.append({
            'Tier':      tier_label,
            'Features':  n_features,
            'Model':     model_name,
            'Precision': round(r['precision'], 4),
            'Recall':    round(r['recall'], 4),
            'F1':        round(r['f1'], 4),
        })
    rows.append({
        'Tier':      tier_label,
        'Features':  n_features,
        'Model':     'XGBoost',
        'Precision': round(xgb_results[tier_name]['precision'], 4),
        'Recall':    round(xgb_results[tier_name]['recall'], 4),
        'F1':        round(xgb_results[tier_name]['f1'], 4),
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

import joblib
joblib.dump(results, 'baseline_results.pkl')
print("\nResults saved to baseline_results.pkl")

 Tier  Features               Model  Precision  Recall     F1
Tier1       130 Logistic Regression     0.3870  0.7076 0.5004
Tier1       130       Random Forest     0.3853  0.6907 0.4947
Tier1       130             XGBoost     0.4606  0.6441 0.5371
Tier2       192 Logistic Regression     0.3737  0.7055 0.4886
Tier2       192       Random Forest     0.3378  0.6992 0.4555
Tier2       192             XGBoost     0.4817  0.6144 0.5400
Tier3       436 Logistic Regression     0.3744  0.6504 0.4752
Tier3       436       Random Forest     0.3984  0.6144 0.4833
Tier3       436             XGBoost     0.4799  0.6568 0.5546
Tier4       833 Logistic Regression     0.3871  0.5742 0.4625
Tier4       833       Random Forest     0.4037  0.6081 0.4852
Tier4       833             XGBoost     0.4754  0.6758 0.5582

Results saved to baseline_results.pkl


In [15]:
import joblib

X_train_tier1, y_train_tier1, X_test_tier1, y_test_tier1 = joblib.load('data_tier1.pkl')

model_best_recall    = joblib.load('tier1_recall_model.pkl')
model_best_precision = joblib.load('tier1_precision_model.pkl')
model_best_f1        = joblib.load('tier1_f1_model.pkl')

print("Models and data loaded")

import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Get binary predictions from each model
pred_recall    = model_best_recall.predict(X_test_tier1)
pred_precision = model_best_precision.predict(X_test_tier1)
pred_f1        = model_best_f1.predict(X_test_tier1)

# Stack predictions and sum votes per student
votes = pred_recall + pred_precision + pred_f1

print("Vote distribution:")
for v in [0, 1, 2, 3]:
    n = (votes == v).sum()
    actual_dropout_rate = y_test_tier1[votes == v].mean()
    print(f"  {v} votes — {n} students — actual dropout rate: {actual_dropout_rate:.3f}")

print()

# Evaluate at each threshold
for min_votes in [1, 2, 3]:
    y_pred_voted = (votes >= min_votes).astype(int)
    report    = classification_report(y_test_tier1, y_pred_voted, output_dict=True)
    f1        = report['1']['f1-score']
    precision = report['1']['precision']
    recall    = report['1']['recall']
    acc       = accuracy_score(y_test_tier1, y_pred_voted)
    print(f"Min votes >= {min_votes}: Accuracy: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

Models and data loaded
Vote distribution:
  0 votes — 2299 students — actual dropout rate: 0.054
  1 votes — 188 students — actual dropout rate: 0.197
  2 votes — 261 students — actual dropout rate: 0.318
  3 votes — 416 students — actual dropout rate: 0.546

Min votes >= 1: Accuracy: 0.7968 | Precision: 0.4012 | Recall: 0.7352 | F1: 0.5191
Min votes >= 2: Accuracy: 0.8328 | Precision: 0.4579 | Recall: 0.6568 | F1: 0.5396
Min votes >= 3: Accuracy: 0.8628 | Precision: 0.5457 | Recall: 0.4809 | F1: 0.5113


In [16]:
from sklearn.metrics import classification_report

for name, model in [('Recall', model_best_recall), ('Precision', model_best_precision), ('F1', model_best_f1)]:
    train_pred = model.predict(X_train_tier1)
    report = classification_report(X_train_tier1 if False else y_train_tier1, train_pred, output_dict=True)
    print(f"{name} — train F1: {report['1']['f1-score']:.4f} | train recall: {report['1']['recall']:.4f}")

Recall — train F1: 0.5503 | train recall: 0.8014
Precision — train F1: 0.9691 | train recall: 0.9958
F1 — train F1: 0.7375 | train recall: 0.9158


In [17]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

tier1_fine_search = joblib.load('tier1_fine_search.pkl')

pred_recall = model_best_recall.predict(X_test_tier1)
pred_f1     = tier1_fine_search.predict(X_test_tier1)

votes = pred_recall + pred_f1

print("Vote distribution:")
for v in [0, 1, 2]:
    n = (votes == v).sum()
    actual_dropout_rate = y_test_tier1[votes == v].mean()
    print(f"  {v} votes — {n} students — actual dropout rate: {actual_dropout_rate:.3f}")

print()

for min_votes in [1, 2]:
    y_pred_voted = (votes >= min_votes).astype(int)
    report    = classification_report(y_test_tier1, y_pred_voted, output_dict=True)
    f1        = report['1']['f1-score']
    precision = report['1']['precision']
    recall    = report['1']['recall']
    acc       = accuracy_score(y_test_tier1, y_pred_voted)
    print(f"Min votes >= {min_votes}: Accuracy: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

Vote distribution:
  0 votes — 2312 students — actual dropout rate: 0.055
  1 votes — 197 students — actual dropout rate: 0.203
  2 votes — 655 students — actual dropout rate: 0.464

Min votes >= 1: Accuracy: 0.7990 | Precision: 0.4038 | Recall: 0.7288 | F1: 0.5196
Min votes >= 2: Accuracy: 0.8360 | Precision: 0.4641 | Recall: 0.6441 | F1: 0.5395


In [18]:
tier1_cols = set(feature_tiers['tier1'])
tier2_cols = set(feature_tiers['tier2'])
tier3_cols = set(feature_tiers['tier3'])
tier4_cols = set(feature_tiers['tier4'])

unique_tier1 = tier1_cols
unique_tier2 = tier2_cols - tier1_cols
unique_tier3 = tier3_cols - tier2_cols
unique_tier4 = tier4_cols - tier3_cols

print(f"TIER 1 ONLY ({len(unique_tier1)} features):")
print(sorted(unique_tier1))

print(f"\nTIER 2 UNIQUE ADDITIONS ({len(unique_tier2)} features):")
print(sorted(unique_tier2))

print(f"\nTIER 3 UNIQUE ADDITIONS ({len(unique_tier3)} features):")
print(sorted(unique_tier3))

print(f"\nTIER 4 UNIQUE ADDITIONS ({len(unique_tier4)} features):")
print(sorted(unique_tier4))

TIER 1 ONLY (130 features):
['A1ALG1LEVELS', 'A1G9ABSENTEE', 'A1G9BEHAVE', 'A1G9BEHIND', 'A1G9CNSLREF', 'A1G9GRADES', 'A1G9OTHER', 'A1G9PRNTREF', 'A1G9REQUEST', 'A1G9TCHREF', 'A1NOTIFY', 'C1ABSENTEE', 'C1ASSIGNMENT', 'C1BEHIND', 'C1CASELOAD', 'C1CNSLREFER', 'C1DISCPROB', 'C1DOPREVOTHR', 'C1G9MSAME', 'C1G9SSAME', 'C1GEDPREP', 'C1PLAN', 'C1PLANPARENT', 'C1POORGRADES', 'C1PRNTREFER', 'C1SIGNOFF', 'C1STUDREQ', 'C1TCHREFER', 'C1UPPERMSAME', 'C1UPPERSSAME', 'M1CERT68', 'M1CERT912', 'M1CERTK5', 'M1CERTTYPE', 'M1HIDEG', 'M1MTHYRS912', 'M1SCHYRS', 'M1TCHYR912', 'M1TCHYRK8', 'P1ADHD', 'P1AUTISM', 'P1CHANGESCH', 'P1DD', 'P1DROPOUT', 'P1EAREYE', 'P1ELLEVER', 'P1ELLNOW', 'P1HOMELANG', 'P1HONORS', 'P1HSSIB', 'P1INTELLECT', 'P1JOINT', 'P1RELSHP', 'P1REPEATG1', 'P1REPEATG2', 'P1REPEATG3', 'P1REPEATG4', 'P1REPEATG5', 'P1REPEATG6', 'P1REPEATG7', 'P1REPEATG8', 'P1REPEATG9', 'P1REPEATGK', 'P1REPEATGRD', 'P1SKIPG1', 'P1SKIPG2', 'P1SKIPG3', 'P1SKIPG8', 'P1SKIPGK', 'P1SKIPGRD', 'P1SLD', 'P1SPECIALED', 'P1SUS

In [19]:
import json

unique_features = {
    'tier1': sorted(list(unique_tier1)),
    'tier2_additions': sorted(list(unique_tier2)),
    'tier3_additions': sorted(list(unique_tier3)),
    'tier4_additions': sorted(list(unique_tier4)),
}

with open('unique_features_per_tier.json', 'w') as f:
    json.dump(unique_features, f, indent=2)

print("Saved to unique_features_per_tier.json")
print(f"\nTier 1:            {len(unique_features['tier1'])} features")
print(f"Tier 2 additions:  {len(unique_features['tier2_additions'])} features")
print(f"Tier 3 additions:  {len(unique_features['tier3_additions'])} features")
print(f"Tier 4 additions:  {len(unique_features['tier4_additions'])} features")

Saved to unique_features_per_tier.json

Tier 1:            130 features
Tier 2 additions:  62 features
Tier 3 additions:  244 features
Tier 4 additions:  397 features
